# Zero-Shot DeBERTa-MNLI Ensemble (0524 突破嘗試)

根據 Schopf et al. 論文，**DeBERTa Zero-Shot Entailment** 是 medical_abstracts 上最強的 unsupervised 方法（57.28 micro F1）。

策略：用此模型對 train（產生 OOF）+ test 做 zero-shot 推論，加進我們的 final_d ensemble。

**為什麼有突破潛力**：之前 ensemble 成員都是「PubMedBERT family 在 train 上 fine-tune」，高度同質；DeBERTa-MNLI 完全沒看過我們的 train labels，提供**真正獨立的 bias 軸**。

**合法性**：使用公開 NLI 預訓練模型 + 公開類別名稱，論文方法，無 label leak。

## Cell 0：環境準備

In [ ]:
import os, sys, shutil

os.chdir('/content')
REPO_DIR = '/content/Data_Mining'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!cd /content && git clone https://github.com/eric20041027/Data_Mining.git
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

!pip install -q -U "transformers>=4.44,<4.50" "accelerate>=0.33" "sentencepiece"

## Cell 1：還原既有訓練 runs（從 Drive）

In [ ]:
import tarfile

candidates = [
    '/content/drive/MyDrive/Kaggle_backup/predictions_FINAL_0524.tar.gz',
    '/content/drive/MyDrive/Kaggle_backup/predictions_deberta_large.tar.gz',
    '/content/drive/MyDrive/Kaggle_backup/predictions_FINAL_0523_complete.tar.gz',
    '/content/drive/MyDrive/Kaggle_backup/predictions_bce_grouped.tar.gz',
]
for tar_path in candidates:
    if os.path.exists(tar_path):
        print(f'Restoring from: {tar_path}')
        with tarfile.open(tar_path) as tar:
            tar.extractall(REPO_DIR)
        print('  done')
        break

src = f'{REPO_DIR}/outputs/bert_runs'
if os.path.isdir(src):
    dirs = sorted(d for d in os.listdir(src) if os.path.isdir(os.path.join(src, d)) and d != 'smoke_test')
    print(f'\n還原後 {len(dirs)} 個 run 資料夾')
    # 確認 final_d + DeBERTa-large 在不在
    needed = ['pubmedbert_noweight_seed42', 'pubmedbert_noweight_seed2024', 
              'biobert_noweight_seed42', 'pubmedbertlarge_noweight_seed42',
              'deberta_v3_large_noweight_seed42']
    for prefix in needed:
        count = sum(1 for d in dirs if d.startswith(prefix))
        status = '✓' if count == 5 else '⚠️'
        print(f'  {status} {prefix}: {count}/5 folds')

## Cell 2：載入 zero-shot 模型 + 定義函式

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 跟我們 LABEL2ID 一致的順序（用 train_bert.py 的 LABEL_LIST）
from utils import LABEL_LIST, NUM_CLASSES, load_train, load_test, make_folds, SEED, OUTPUTS_DIR
print('Internal label order (NLI prompts will follow this):')
for i, n in enumerate(LABEL_LIST):
    print(f'  idx {i}: {n}')

# Schopf et al. 論文使用的 hypothesis template (page 4)
# "For the Medical Abstracts dataset just the class names are used as hypotheses/label descriptions."
hypothesis_template = "This text is about {}."
hypotheses = [hypothesis_template.format(name) for name in LABEL_LIST]
print('\nHypotheses:')
for h in hypotheses:
    print(f'  {h}')

In [ ]:
MODEL_NAME = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-docnli-ling-2c'
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to('cuda').eval()
model = model.to(torch.bfloat16)  # A100 supports bf16, faster inference
print('Model loaded.')
print(f'NLI label config: {model.config.id2label}')

In [ ]:
# 此模型是 2-class NLI: 0=entailment, 1=not_entailment
# 對每個 (text, hypothesis) pair 做 forward，取 entailment 機率，再 softmax across 5 hypotheses

ENTAIL_IDX = None
for idx, name in model.config.id2label.items():
    if 'entail' in name.lower():
        ENTAIL_IDX = idx
        break
assert ENTAIL_IDX is not None, f'No entailment class found in {model.config.id2label}'
print(f'Entailment label index: {ENTAIL_IDX}')

@torch.no_grad()
def predict_zero_shot(texts, batch_size=16, max_length=512):
    """Return (N, 5) array of class probabilities via NLI entailment."""
    all_probs = []
    n = len(texts)
    for i in range(0, n, batch_size):
        batch_texts = texts[i:i+batch_size]
        # For each text, create 5 (text, hypothesis) pairs
        premise_list = []
        hypothesis_list = []
        for t in batch_texts:
            for h in hypotheses:
                premise_list.append(t)
                hypothesis_list.append(h)
        # Batch-encode all pairs (batch_size * 5 pairs)
        enc = tokenizer(
            premise_list, hypothesis_list,
            truncation='only_first',  # only truncate premise (text), keep hypothesis
            max_length=max_length,
            padding=True,
            return_tensors='pt'
        ).to('cuda')
        logits = model(**enc).logits.float()  # (B*5, 2 or 3)
        # Get entailment logit
        entail_logits = logits[:, ENTAIL_IDX]  # (B*5,)
        # Reshape to (B, 5) and softmax
        entail_logits = entail_logits.view(len(batch_texts), len(hypotheses))
        probs = F.softmax(entail_logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        if (i // batch_size) % 50 == 0:
            print(f'  {i + len(batch_texts)}/{n} ({100*(i+len(batch_texts))/n:.1f}%)')
    return np.vstack(all_probs)

# Smoke test
test_probs = predict_zero_shot(['A study of breast cancer in 50 patients showed metastases to the liver.'])
print(f'\nSmoke test prob: {test_probs[0].round(3)}')
print(f'Argmax: {LABEL_LIST[test_probs[0].argmax()]}')
# 預期：neoplasms (idx 0) 機率最高

## Cell 3：對 train + test 做 zero-shot 推論（~15 分鐘）

In [ ]:
import time

train = load_train()
test = load_test()

print(f'Predicting train (n={len(train)})...')
t0 = time.time()
train_probs = predict_zero_shot(train['condition'].tolist(), batch_size=12)
print(f'Train done in {time.time()-t0:.0f}s, shape={train_probs.shape}')

print(f'\nPredicting test (n={len(test)})...')
t0 = time.time()
test_probs = predict_zero_shot(test['condition'].tolist(), batch_size=12)
print(f'Test done in {time.time()-t0:.0f}s, shape={test_probs.shape}')

In [ ]:
# 驗證 OOF macro F1
from sklearn.metrics import f1_score, classification_report

train_pred = train_probs.argmax(axis=1)
oof_f1 = f1_score(train['label_idx'], train_pred, average='macro')
print(f'Zero-shot OOF Macro F1 on train: {oof_f1:.4f}')
print('\nClassification report:')
print(classification_report(train['label_idx'], train_pred, target_names=LABEL_LIST, digits=4))

print(f'\nTest submission dist (預測分布):')
test_pred = test_probs.argmax(axis=1)
print(pd.Series(test_pred).value_counts(normalize=True).sort_index().round(3))
print('\n論文報告 (micro F1) 57.28 — 我們的 macro 通常會略低')

## Cell 4：以「fake run dirs」格式儲存，相容 ensemble_predict.py

In [ ]:
import json
from pathlib import Path

# Zero-shot 沒有訓練概念，沒有「per fold val」。但為了相容 ensemble_predict.py，
# 我們把同一份預測切成 5 fold（按 StratifiedKFold seed=42 的 fold assignment）
train_folded = make_folds(train, seed=SEED).reset_index(drop=True)

TAG_PREFIX = 'zero_shot_deberta_mnli'
for fold in range(5):
    run_dir = Path(REPO_DIR) / 'outputs' / 'bert_runs' / f'{TAG_PREFIX}_fold{fold}'
    run_dir.mkdir(parents=True, exist_ok=True)
    
    # 該 fold 的 val 對應的 train rows
    fold_mask = train_folded['fold'] == fold
    fold_val_probs = train_probs[fold_mask.values]
    
    np.save(run_dir / 'val_probs.npy', fold_val_probs)
    np.save(run_dir / 'test_probs.npy', test_probs)
    
    # 寫 args.json 跟 metrics.json（ensemble_predict 需要 args.json 的 fold 欄位）
    args = {
        'fold': fold,
        'seed': SEED,
        'model': MODEL_NAME,
        'method': 'zero_shot_NLI_entailment',
        'hypothesis_template': hypothesis_template,
        'labels': LABEL_LIST,
    }
    (run_dir / 'args.json').write_text(json.dumps(args, indent=2))
    
    fold_pred = fold_val_probs.argmax(axis=1)
    fold_truth = train_folded.loc[fold_mask, 'label_idx'].values
    fold_f1 = f1_score(fold_truth, fold_pred, average='macro')
    metrics = {
        'fold': fold,
        'val_macro_f1': float(fold_f1),
        'method': 'zero_shot_NLI',
        'n_val': int(fold_mask.sum()),
    }
    (run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    print(f'fold {fold}: n={int(fold_mask.sum())}, val Macro F1 = {fold_f1:.4f} -> saved to {run_dir.name}')

print('\n✅ Zero-shot 預測已存成 5 個 fake run dirs，相容 ensemble_predict.py')

## Cell 5：跑各種 ensemble 候選

In [ ]:
import subprocess

def run_ensemble(patterns, tag, prefer_tta=False):
    cmd = ['python', 'src/ensemble_predict.py', '--bert-runs'] + patterns + ['--no-overlap-constraint', '--tag', tag]
    if prefer_tta:
        cmd.append('--prefer-tta')
    print('>>>', ' '.join(cmd))
    out = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        print('STDERR:', out.stderr[:500])
        return
    for line in out.stdout.split('\n'):
        if any(k in line for k in ['Found ', 'used TTA', 'fold ', 'OOF Macro F1', 'macro avg', 'general path']):
            print(line)
    print()

# v26: final_d (4 model) + zero-shot DeBERTa-MNLI（不含 supervised DeBERTa-large）
print('=== v26: final_d + zero-shot DeBERTa-MNLI ===')
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
    f'outputs/bert_runs/{TAG_PREFIX}_fold*',
], 'v26_final_d_plus_zeroshot')

# v27: final_d + supervised DeBERTa-large + zero-shot（最完整）
print('=== v27: full ensemble (含 supervised DeBERTa-large + zero-shot) ===')
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
    'outputs/bert_runs/deberta_v3_large_noweight_seed*_fold*',
    f'outputs/bert_runs/{TAG_PREFIX}_fold*',
], 'v27_full_with_zeroshot')

# v28: 同 v27 但 DeBERTa-large 用 TTA
print('=== v28: 同 v27 但 DeBERTa-large 用 TTA ===')
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
    'outputs/bert_runs/pubmedbert_noweight_seed2024_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
    'outputs/bert_runs/deberta_v3_large_noweight_seed*_fold*',
    f'outputs/bert_runs/{TAG_PREFIX}_fold*',
], 'v28_full_with_zeroshot_tta', prefer_tta=True)

# v29: 純 zero-shot 看 baseline
print('=== v29: 純 zero-shot DeBERTa-MNLI ===')
run_ensemble([
    f'outputs/bert_runs/{TAG_PREFIX}_fold*',
], 'v29_zeroshot_only')

## Cell 6：對比 v26/v27/v28 跟 final_d / v25

In [ ]:
import hashlib

submissions = {}
for tag in ['v26_final_d_plus_zeroshot', 'v27_full_with_zeroshot', 
            'v28_full_with_zeroshot_tta', 'v29_zeroshot_only']:
    p = f'{REPO_DIR}/outputs/submission_{tag}.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        h = hashlib.md5(open(p, 'rb').read()).hexdigest()[:10]
        dist = df['label'].value_counts(normalize=True).sort_index().round(3).to_dict()
        submissions[tag] = (h, dist)
        print(f'{tag}: md5={h} dist={dist}')

# 也對比 final_d（如果有）
for fd_path in [
    f'{REPO_DIR}/outputs/submission_final_d_4noweight.csv',
    f'{REPO_DIR}/outputs/submissions/submission_final_d_4noweight.csv',
]:
    if os.path.exists(fd_path):
        fd = pd.read_csv(fd_path)
        print(f'\nfinal_d (LB 0.646) dist: {fd["label"].value_counts(normalize=True).sort_index().round(3).to_dict()}')
        for tag, _ in submissions.items():
            new = pd.read_csv(f'{REPO_DIR}/outputs/submission_{tag}.csv')
            diff = (new['label'] != fd['label']).sum()
            print(f'  {tag} vs final_d: {diff} 筆不同')
        break

## Cell 7：下載 + 備份

In [ ]:
from google.colab import files

for tag in ['v26_final_d_plus_zeroshot', 'v27_full_with_zeroshot', 
            'v28_full_with_zeroshot_tta', 'v29_zeroshot_only']:
    p = f'{REPO_DIR}/outputs/submission_{tag}.csv'
    if os.path.exists(p):
        files.download(p)

# 最終備份（含 zero-shot fake runs）
import tarfile, glob
src = f'{REPO_DIR}/outputs/bert_runs'
dst = '/content/drive/MyDrive/Kaggle_backup/predictions_with_zeroshot.tar.gz'
files_to_pack = []
for run in sorted(os.listdir(src)):
    rd = os.path.join(src, run)
    if not os.path.isdir(rd): continue
    for pat in ['*.npy', '*.json', '*.csv']:
        files_to_pack += glob.glob(os.path.join(rd, pat))
with tarfile.open(dst, 'w:gz') as tar:
    for f in files_to_pack:
        tar.add(f, arcname=os.path.relpath(f, REPO_DIR))
print(f'\nBackup: {dst}')

## 🎯 提交決策

看上面 v26 / v27 / v28 / v29 的 OOF：

| 結果 | 意義 | 建議提交 |
|---|---|---|
| v27 / v28 OOF ≥ 0.665 | zero-shot 真的有貢獻 | 主交 v28 (含 TTA) |
| v26 OOF ≥ 0.665 | 不需 supervised DeBERTa-large | 主交 v26 |
| v29 OOF ≥ 0.55 | zero-shot 單獨還算強 | – |
| 全部沒超過 final_d (0.6592) | zero-shot 對 ensemble 沒幫助 | 不提交 |

**預期**：v26 OOF 應該 0.66-0.67（如果 zero-shot 真的提供互補訊號），對應 LB 0.65-0.66（+0.005-0.015 over final_d）